In [372]:
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import os 

load_dotenv()
login(os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [28]:
from datasets import enable_progress_bar
enable_progress_bar()
dataset = load_dataset(
    "bigcode/starcoderdata",
    data_dir="jupyter-structured-clean-dedup",
    split="train",
    streaming
)

Process ForkServerPoolWorker-29:
Process ForkServerPoolWorker-30:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib64/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib64/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.14/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/usr/lib64/python3.14/multiprocessing/queues.py", line 385, in get
    res = self._reader.recv_bytes()
  File "/usr/lib64/python3.14/multiprocessing/connection.py", line 226, in recv_bytes
    buf = self._recv_bytes(maxlength)
  File "/usr/lib64/python3.14/multiprocessing/connection.py", line 451, in _recv_bytes
    buf = self._recv(4)
  File "/usr/lib64/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib64/python3.14/m

KeyboardInterrupt: 

In [3]:
import re, ast

In [4]:
"""
Strip Jupyter magic syntax from a code cell and validate the remainder
with ast.parse, without corrupting ordinary code (modulus operator,
string contents, etc.).
"""

import ast
import io
import tokenize


def _protected_lines(source: str) -> set[int]:
    """
    Return the set of 1-indexed line numbers that lie inside a
    multi-line string token (e.g. a triple-quoted string body).
    These lines must never be treated as magic lines, even if their
    stripped text happens to start with '%' or '!'.
    """
    protected: set[int] = set()
    try:
        tokens = tokenize.generate_tokens(io.StringIO(source).readline)
        for tok in tokens:
            if tok.type == tokenize.STRING:
                start_row, _ = tok.start
                end_row, _ = tok.end
                if end_row > start_row:
                    # Protect every row from just after the opening
                    # quote through the closing quote. The start_row
                    # itself is not blanket-protected because it also
                    # contains real code before the string literal
                    # begins (e.g. `s = """`).
                    protected.update(range(start_row + 1, end_row + 1))
    except tokenize.TokenError:
        # Tokenizing failed outright; fall back to no protection and
        # let ast.parse's error surface normally.
        pass
    return protected


def _is_magic_line(line: str) -> bool:
    stripped = line.lstrip()
    return stripped.startswith("%%") or stripped.startswith("%") or stripped.startswith("!")


def strip_magics(source: str) -> str:
    """
    Remove Jupyter magic lines from `source`, leaving all other lines
    (including magic-looking text inside multi-line strings) intact.
    Blank lines are substituted in place of removed magics so that
    line numbers of the remaining code are preserved.
    """
    lines = source.splitlines(keepends=True)
    if not lines:
        return source

    # A '%%' on the first non-blank line is a *cell* magic: by Jupyter
    # convention it consumes the entire remainder of the cell as its
    # argument, which is not Python source at all.
    for line in lines:
        if line.strip() == "":
            continue
        if line.lstrip().startswith("%%"):
            return ""
        break

    protected = _protected_lines(source)

    while True:
        candidate = "".join(lines)
        try:
            ast.parse(candidate)
            return candidate
        except SyntaxError as err:
            lineno = err.lineno
            if lineno is None or lineno > len(lines):
                raise
            idx = lineno - 1
            line = lines[idx]

            if lineno in protected or not _is_magic_line(line):
                # Not something we can attribute to a magic: a real
                # syntax error. Stop and let the caller see it.
                raise SyntaxError

            # Preserve the trailing newline so line numbers of
            # subsequent lines are unaffected.
            lines[idx] = "\n" if line.endswith("\n") else ""


def validate_cell(source: str) -> tuple[bool, str]:
    """
    Strip magics from `source` and check the result with ast.parse.
    Returns (is_valid, message). On success, message is the cleaned
    source. On failure, message is the SyntaxError description.
    """
    try:
        cleaned = strip_magics(source)
    except SyntaxError as err:
        return False, f"line {err.lineno}: {err.msg}"

    try:
        ast.parse(cleaned)
        return True, cleaned
    except SyntaxError as err:
        return False, f"line {err.lineno}: {err.msg}"


def clean_cell(source: str) -> str | None:
    """
    Return magic-free source suitable for a training corpus, or None
    if the cell contributes no usable standalone code:
      - the cell was a %% cell magic (entire cell dropped),
      - remaining code is blank/whitespace-only after stripping,
      - the remaining code still fails ast.parse (e.g. it relied on
        notebook-only state, such as a bare `!ls` with no surrounding
        Python, or a cell that only reads clean up to a real error).
    """
    is_valid, cleaned = validate_cell(source)
    if not is_valid:
        return None
    if cleaned.strip() == "":
        return None
    return cleaned

In [5]:
cell_pattern = re.compile(
    r'(<jupyter_start>|<jupyter_text>|<jupyter_code>|<jupyter_output>|<empty_output>)'  # Group 1 (Tag)
    r'(.*?)'                                                                             # Group 2 (Content)
    r'(?=<jupyter_start>|<jupyter_text>|<jupyter_code>|<jupyter_output>|<empty_output>|$)', # Lookahead
    re.DOTALL
)

In [18]:
def clean_notebook_(file):
    notebook = file["content"]
    cells = cell_pattern.finditer(notebook)
    chunks = []
    for cell in cells:
        cell_type = cell.group(1)
        cell_content = cell.group(2)

        if cell_type == "<jupyter_code>":
            cleaned_code = clean_cell(cell_content)
            if cleaned_code is not None:
                chunks.append(cleaned_code)
            else:
                return None
                
    return {"content": "\n\n".join(chunks)}
    

In [7]:
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
ds = dataset.map(clean_notebook_)
for i, sample in enumerate(ds):
    print(sample["content"])
    print('-----------------------------')
    if i == 10:
        break

In [8]:
import polars as pl

In [9]:
data_dir = "data/jupyter"
import os

os.makedirs(data_dir, exist_ok=True)

In [ ]:
def clean_notebook(notebook):
    cells = cell_pattern.finditer(notebook)
    chunks = []
    for cell in cells:
        cell_type = cell.group(1)
        cell_content = cell.group(2)

        if cell_type == "<jupyter_code>":
            cleaned_code = clean_cell(cell_content)
            if cleaned_code is not None:
                chunks.append(cleaned_code)
            else:
                return None
                
    return "\n\n".join(chunks)
    

In [368]:
def clean_jupyter(file):
    if file["lang"] != python:
        return {"content": None}
    content = file["content"]
    chunks = []
    nb = reads(content, fmt="py:percent")
    
    for cell in nb.cells:
        if cell.cell_type == "code":
            cleaned = clean_cell(cell.source)
            if cleaned is not None:
                chunks.append(cleaned)
                
    return "\n\n".join(chunks)
    

In [ ]:
iterator = iter(dataset)

In [ ]:
file = next(iterator)
notebook = file["content"]
print(clean_notebook(notebook))

In [ ]:
buffer = []
batch_size = 100000
file_counter = 0

for file in tqdm(dataset, desc="Processing notebooks..."):
    if file.get("size", float("inf")) > 1024**2:
        continue
    notebook = file["content"]
    cleaned = clean_notebook(notebook)
    if cleaned is not None:
        buffer.append(cleaned)
        if len(buffer) >= batch_size:
            df = pl.DataFrame(buffer)
            path = f"{data_dir}/file_{file_counter}.parquet"
            df.write_parquet(path)
            file_counter += 1
            buffer.clear()
            print(f"Created file {file_counter}")
if buffer:
    df = pl.DataFrame(buffer)
    df.write_parquet(f"{data_dir}/file_{file_counter}.parquet")
    buffer.clear()

In [50]:
buffer = [clean_notebook(next(iterator)["content"]) for _ in range(10)]
df = pl.DataFrame(buffer)
df

column_0
str
"""#Import Pandas Library import …"
null
"""a=12 b=100 if a>b: print(""a …"
"""# Imports # code from https:/…"
null
"""import pandas as pd df = pd.re…"
"""from tensorflow.python.client …"
null
"""# import import numpy as np i…"


In [32]:
import os
import tempfile
import polars as pl
from concurrent.futures import ProcessPoolExecutor, as_completed
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm.auto import tqdm

REPO_ID = "bigcode/starcoderdata"
FOLDER_NAME = "jupyter-structured-clean-dedup"

# 1. Automatically find all parquet shard files inside that specific folder
all_files = list_repo_files(repo_id=REPO_ID, repo_type="dataset")
shard_filenames = [f for f in all_files if f.startswith(FOLDER_NAME) and f.endswith(".parquet")]

print(f"Found {len(shard_filenames)} shards to process.")

Found 6 shards to process.


In [329]:
dataset = load_dataset(
    "bigcode/starcoderdata",
    data_dir="jupyter-scripts-dedup-filtered",
    split="train",
    streaming=True
)
iterator = iter(dataset)

In [332]:
import jupytext 
from jupytext import reads

{'hexsha': '1f62eb8b7460292d191a23eead9c13533a5825bd', 'ext': 'jl', 'lang': 'julia', 'max_stars_repo_path': 'notebooks/SHA-RNN.ipynb', 'max_stars_repo_name': 'alisafaya/SHA-RNN.jl', 'max_stars_repo_licenses': "['MIT']", 'avg_line_length': 30.698924731182796, 'alphanum_fraction': 0.6736770691994572, 'size': 2948, 'id': '133', 'content': '# ---\n# jupyter:\n#   jupytext:\n#     text_representation:\n#       extension: .jl\n#       format_name: light\n#       format_version: \'1.5\'\n#       jupytext_version: 1.14.4\n#   kernelspec:\n#     display_name: Julia 1.3.1\n#     language: julia\n#     name: julia-1.3\n# ---\n\n# +\n@info "Train baseline Single Headed Attention Recurrent language model using enwik8 dataset..."\n@info "This model is the main model of SHA-RNN, which contains 4 layers of SHA-RNN"\nusing Knet\n\ninclude("../src/data.jl")\ninclude("../src/model.jl")\ninclude("../src/train.jl")\n\n# +\n\nBATCHSIZE = 2 ; @show BATCHSIZE\nBPTT = 1024 ; @show BPTT\nMEMSIZE = 5000 ; @show 

In [373]:
ds = load_dataset("bigcode/starcoderdata", data_dir="python", split="train", streaming=True)
iterator = iter(ds)

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [374]:
next(iterator)

{'max_stars_repo_path': 'public_data/serializers.py',
 'max_stars_repo_name': 'MTES-MCT/sparte',
 'max_stars_count': 0,
 'id': '0',
 'content': '<reponame>MTES-MCT/sparte\nfrom rest_framework_gis import serializers\nfrom rest_framework import serializers as s\n\nfrom .models import (\n    Artificialisee2015to2018,\n    Artificielle2018,\n    CommunesSybarval,\n    CouvertureSol,\n    EnveloppeUrbaine2018,\n    Ocsge,\n    Renaturee2018to2015,\n    Sybarval,\n    Voirie2018,\n    ZonesBaties2018,\n    UsageSol,\n)\n\n\ndef get_label(code="", label=""):\n    if code is None:\n        code = "-"\n    if label is None:\n        label = "inconnu"\n    return f"{code} {label[:30]}"\n\n\nclass Artificialisee2015to2018Serializer(serializers.GeoFeatureModelSerializer):\n    usage_2015 = s.SerializerMethodField()\n    usage_2018 = s.SerializerMethodField()\n    couverture_2015 = s.SerializerMethodField()\n    couverture_2018 = s.SerializerMethodField()\n\n    def get_usage_2015(self, obj):\n    

In [378]:
import ast

# Passing an instantiated node incorrectly to parse, 
# or leaving required identifier fields as None
node = ast.Name(id=None, ctx=ast.Load()) 
ast.parse(node)  # Throws ValueError or TypeError


TypeError: expected Module node, got Name